In [2]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/techieyash/vyomai-ai4i-cooling-final-dataset/cooling_telemetry.csv
/kaggle/input/datasets/techieyash/vyomai-ai4i-cooling-final-dataset/ai4i2020.csv


In [3]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

In [4]:
AI4I_PATH = "/kaggle/input/datasets/techieyash/vyomai-ai4i-cooling-final-dataset/ai4i2020.csv"
COOLING_PATH = "/kaggle/input/datasets/techieyash/vyomai-ai4i-cooling-final-dataset/cooling_telemetry.csv"

ai4i = pd.read_csv(AI4I_PATH)
cooling = pd.read_csv(COOLING_PATH)

print("AI4I:", ai4i.shape)
print("Cooling:", cooling.shape)

print(ai4i.columns.tolist())
print(cooling.columns.tolist())

AI4I: (10000, 14)
Cooling: (64800, 7)
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
['device_id', 'timestamp', 'fan_rpm', 'bearing_vibration_mm_s', 'airflow_cfm', 'label', 'days_to_failure']


In [5]:
def map_failure(row):
    if row["HDF"] == 1:
        return "Thermal Failure"
    elif row["PWF"] == 1:
        return "Power Failure"
    elif row["OSF"] == 1:
        return "Operational Failure"
    elif row["TWF"] == 1:
        return "Operational Failure"
    else:
        return "Healthy"

ai4i["failure_type"] = ai4i.apply(map_failure, axis=1)

print(ai4i["failure_type"].value_counts())

failure_type
Healthy                9670
Operational Failure     123
Thermal Failure         115
Power Failure            92
Name: count, dtype: int64


In [6]:
ai4i = ai4i.copy()

# Drop ID columns
ai4i = ai4i.drop(columns=["UDI", "Product ID"], errors="ignore")

# Convert Kelvin to Celsius
ai4i["air_temp_c"] = ai4i["Air temperature [K]"] - 273.15
ai4i["process_temp_c"] = ai4i["Process temperature [K]"] - 273.15

# Rename useful columns
ai4i["rotational_speed_rpm"] = ai4i["Rotational speed [rpm]"]
ai4i["torque_nm"] = ai4i["Torque [Nm]"]
ai4i["tool_wear_min"] = ai4i["Tool wear [min]"]

# Engineered features
ai4i["temp_delta_c"] = ai4i["process_temp_c"] - ai4i["air_temp_c"]
ai4i["mechanical_load"] = ai4i["torque_nm"] * ai4i["rotational_speed_rpm"]
ai4i["power_proxy"] = ai4i["torque_nm"] * ai4i["rotational_speed_rpm"] / 9550
ai4i["wear_torque_ratio"] = ai4i["tool_wear_min"] / (ai4i["torque_nm"] + 1)

# Encode product type
type_map = {"L": 0, "M": 1, "H": 2}
ai4i["type_encoded"] = ai4i["Type"].map(type_map).fillna(0)

ai4i_feature_cols = [
    "air_temp_c",
    "process_temp_c",
    "temp_delta_c",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min",
    "mechanical_load",
    "power_proxy",
    "wear_torque_ratio",
    "type_encoded"
]

X = ai4i[ai4i_feature_cols]
y_text = ai4i["failure_type"]

print(X.head())
print(y_text.value_counts())

   air_temp_c  process_temp_c  temp_delta_c  rotational_speed_rpm  torque_nm  \
0       24.95           35.45          10.5                  1551       42.8   
1       25.05           35.55          10.5                  1408       46.3   
2       24.95           35.35          10.4                  1498       49.4   
3       25.05           35.45          10.4                  1433       39.5   
4       25.05           35.55          10.5                  1408       40.0   

   tool_wear_min  mechanical_load  power_proxy  wear_torque_ratio  \
0              0          66382.8     6.951079           0.000000   
1              3          65190.4     6.826220           0.063425   
2              5          74001.2     7.748817           0.099206   
3              7          56603.5     5.927068           0.172840   
4              9          56320.0     5.897382           0.219512   

   type_encoded  
0             1  
1             0  
2             0  
3             0  
4             

In [7]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

print("Label mapping:")
for cls, code in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(cls, "->", code)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Label mapping:
Healthy -> 0
Operational Failure -> 1
Power Failure -> 2
Thermal Failure -> 3
Train: (8000, 10)
Test: (2000, 10)


In [8]:
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

model_ai4i = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

model_ai4i.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

print("AI4I balanced model trained")

AI4I balanced model trained


In [9]:
y_pred = model_ai4i.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9765
                     precision    recall  f1-score   support

            Healthy       1.00      0.98      0.99      1934
Operational Failure       0.35      0.68      0.46        25
      Power Failure       0.86      1.00      0.92        18
    Thermal Failure       0.88      0.96      0.92        23

           accuracy                           0.98      2000
          macro avg       0.77      0.90      0.82      2000
       weighted avg       0.98      0.98      0.98      2000

[[1896   32    3    3]
 [   8   17    0    0]
 [   0    0   18    0]
 [   1    0    0   22]]


In [10]:
import pickle

with open("ai4i_failure_model.pkl", "wb") as f:
    pickle.dump(model_ai4i, f)

with open("ai4i_features.pkl", "wb") as f:
    pickle.dump(ai4i_feature_cols, f)

with open("ai4i_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("AI4I model saved")

AI4I model saved


In [11]:
import os

print(os.listdir("/kaggle/working"))

['ai4i_features.pkl', 'ai4i_failure_model.pkl', '.virtual_documents', 'ai4i_label_encoder.pkl']
